In [21]:
import numpy as np 
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import math
import matplotlib as mpl 
from mosqlient.scoring import compute_wis

state_names = {
    11:"RO", 12:"AC", 13:"AM", 14:"RR", 15:"PA", 16:"AP", 17:"TO",
    21:"MA", 22:"PI", 23:"CE", 24:"RN", 25:"PB", 26:"PE", 27:"AL", 28:"SE", 29:"BA",
    31:"MG", 32:"ES", 33:"RJ", 35:"SP",
    41:"PR", 42:"SC", 43:"RS",
    50:"MS", 51:"MT", 52:"GO", 53:"DF"
}

mpl.rcParams['axes.edgecolor'] = 'gray'

# Definir a cor das linhas dos ticks maiores e menores como cinza
mpl.rcParams['xtick.color'] = 'gray'
mpl.rcParams['ytick.color'] = 'gray'
mpl.rcParams['xtick.labelcolor'] = 'black'
mpl.rcParams['ytick.labelcolor'] = 'black'
plt.rcParams['axes.labelsize'] = 16 # Axis labels
plt.rcParams['xtick.labelsize'] = 14  # X-axis tick labels
plt.rcParams['ytick.labelsize'] = 14  # Y-axis tick labels
plt.rcParams['font.size'] = 16  # General font size
FONT = 12

In [3]:
challenge = 'dengue_state'

df_preds = pd.read_csv(f'../predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds = df_preds.loc[df_preds.adm_1 != 32]
df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_1,id,validation,wis,model
0,2022-10-09,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,13.000000,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
1,2022-10-16,37.019327,43.295517,51.743385,66.653189,83.207109,100.790847,116.992117,126.864602,134.406880,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
2,2022-10-23,55.945220,68.411390,84.200302,108.887826,135.698419,163.344639,189.017119,205.765306,218.575663,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
3,2022-10-30,58.185781,69.697014,83.792687,107.076328,132.601767,158.745145,184.276095,201.204977,212.912726,12,6822,1,95.67,3rd_imdc_isi_isi-dengue
4,2022-11-06,58.634062,68.591352,80.855678,101.969117,125.548661,150.259612,174.517426,190.524410,202.512661,12,6822,1,95.67,3rd_imdc_isi_isi-dengue


In [4]:
def get_data(name = 'dengue_state'): 

    df = pd.read_csv(f'../data/{name}.csv.gz')

    return df

In [5]:
df_dengue_state = get_data('dengue_state')

df_dengue_state.date = pd.to_datetime(df_dengue_state.date)

df_dengue_state = df_dengue_state.loc[df_dengue_state.adm_1 != 32]


df_dengue_state.head()

,date,adm_1,casos
0,2010-01-03,11,2456
1,2010-01-03,12,760
2,2010-01-03,13,155
3,2010-01-03,14,25
4,2010-01-03,15,115


Ranks: 

In [6]:
df_rk_1 = pd.read_csv('default_rk_1.csv.gz')
df_rk_2 = pd.read_csv('norm_rk_2.csv.gz')
df_rk_3 = pd.read_csv('diff_rk_3.csv.gz')
df_rk_4 = pd.read_csv('mean_rk_4.csv.gz')

In [7]:
def get_wis_model(df_preds, df_dengue_state, df_rk_1, label = 'rk_1', rank = 5):

    df_rk_1 = df_rk_1.loc[df_rk_1.adm_1 != 32] 

    list_dfs = [] 

    for adm in df_rk_1.adm_1.unique(): 


        models = df_rk_1.loc[(df_rk_1.adm_1 == adm) & (df_rk_1['rank'] <= rank)].model.unique()

        df_ = df_preds.loc[(df_preds.adm_1 == adm) & ( df_preds.model.isin(models))].groupby(['date', 'adm_1', 'validation'])[['lower_95', 'lower_90', 'lower_80', 'lower_50', 'pred',
                'upper_50', 'upper_80', 'upper_90', 'upper_95']].median().reset_index()

        df_m=df_.merge(df_dengue_state, on = ['date', 'adm_1']).sort_values(by = 'date')


        for val in [1,2,3,4]: 

                wis = np.mean(compute_wis(
                df = df_m.loc[df_m.validation == val][['date', 'lower_95', 'lower_90', 'lower_80', 'lower_50', 'pred',
                        'upper_50', 'upper_80', 'upper_90', 'upper_95']],
                observed_value=  df_m.loc[df_m.validation == val]['casos']))


                list_dfs.append(pd.DataFrame([[adm, val, wis]], columns = ['adm_1', 'validation', 'WIS' ]))



    df_wis_ = pd.concat(list_dfs, ignore_index=True)

    df_wis_['rank'] = label

    return df_wis_

    

In [8]:
df_wis_end = pd.DataFrame()

for df_rank, label in zip([df_rk_1, df_rk_2, df_rk_3, df_rk_4], ['rk_1', 'rk_2', 'rk_3', 'rk_4']): 

    for r in np.arange(1,27): 
        df_wis = get_wis_model(df_preds, df_dengue_state, df_rank, label = label, rank = r)
        df_wis['models_included'] = r

        df_wis_end = pd.concat([df_wis_end, df_wis], ignore_index = True)

df_wis_end.head()

,adm_1,validation,WIS,rank,models_included
0,14,1,2.190290,rk_1,1
1,14,2,3.523752,rk_1,1
2,14,3,2.724626,rk_1,1
3,14,4,3.961659,rk_1,1
4,28,1,14.014227,rk_1,1


In [9]:
df_wis_end['rank'].unique()

<ArrowStringArray>
['rk_1', 'rk_2', 'rk_3', 'rk_4']
Length: 4, dtype: str

In [18]:
df_wis_end.adm_1.unique()

array([14, 28, 27, 13, 24, 25, 42, 33, 43, 41, 31, 35, 29, 53, 15, 51, 52,
       16, 17, 22, 11, 50, 12, 21, 26, 23])

In [37]:
state = 41


for state in df_wis_end.adm_1.unique(): 

    _,ax = plt.subplots(2,2, figsize = (12, 7), sharex = True)

    for ax_, val in zip(ax.ravel(), np.arange(1,5)): 

        df_ = df_wis_end.loc[(df_wis_end.adm_1 == state) & (df_wis_end.validation == val)]

        sns.lineplot(data=df_, x= 'models_included', y = 'WIS', hue = 'rank', ax = ax_)

        ax_.set_title(f'Validation: {val}')

        ax_.axvline(df_.loc[df_.WIS == df_.WIS.min()]['models_included'].values[0], color = 'black', ls  = '--')

        if ax_.legend_ is not None:
            ax_.legend_.remove()

    handles, labels = ax_.get_legend_handles_labels()
    _.legend(
        handles,
        labels,
        loc="upper center",
        ncol=len(labels),
        bbox_to_anchor=(0.5, 0.98 )
    )
    plt.suptitle(state_names[state], y= 1.01)
    plt.tight_layout()

    plt.savefig(f'figures/wis_time_{state}.png', dpi = 300, bbox_inches = 'tight')
    plt.close()


In [11]:
df_

,adm_1,validation,WIS,rank,models_included
39,41,4,523.118796,rk_1,1
143,41,4,904.648937,rk_1,2
247,41,4,670.297671,rk_1,3
351,41,4,753.920821,rk_1,4
455,41,4,659.672312,rk_1,5
...,...,...,...,...,...
10375,41,4,554.498326,rk_4,22
10479,41,4,546.670429,rk_4,23
10583,41,4,519.942153,rk_4,24
10687,41,4,509.334685,rk_4,25


np.int64(15)

In [12]:
df_.WIS.min()

np.float64(320.19570938551755)